In [ ]:
    ############    #############   Separation of Concerns   #############   ##############   

 =>  Keep distinct kinds of change in distinct places: what the app does (business logic),
       how it's configured (environment), how it's deployed (infra/process), and how it
       talks to the network (transport) shouldn't be tangled together in one file.

 =>  The 12-factor app methodology (originally from Heroku, still the industry reference)
       is separation of concerns turned into 12 concrete, checkable rules for portable,
       cloud-ready services.


| # | Factor | In one line |
|---|---|---|
| 1 | Codebase | One codebase in version control, many deploys from it |
| 2 | Dependencies | Explicitly declared and isolated (requirements.txt / pyproject.toml, a venv) |
| 3 | Config | Stored in the environment, never hard-coded (see Phase 0.1's Configuration notebook) |
| 4 | Backing services | Treat a DB/queue/cache as an attached resource, swappable via config |
| 5 | Build, release, run | Strictly separate these three stages |
| 6 | Processes | Stateless, share-nothing -- any state lives in a backing service, not memory |
| 7 | Port binding | The app is self-contained and exports itself via a port (no runtime web-server injection) |
| 8 | Concurrency | Scale out via multiple processes, not one process trying to do everything |
| 9 | Disposability | Fast startup, graceful shutdown -- instances are disposable, not precious |
| 10 | Dev/prod parity | Keep dev, staging, prod as similar as possible |
| 11 | Logs | Treat logs as an event stream (stdout), not a file the app manages itself |
| 12 | Admin processes | One-off admin/maintenance tasks run the same way as the app itself |


In [ ]:
# Factor 6 (stateless processes) violated vs fixed -- a classic bug class

# BAD: in-memory state means a user's session breaks the moment a request hits a
# different process/instance (common the moment you scale beyond 1 instance)
class BadSessionStore:
    def __init__(self):
        self._sessions: dict[str, dict] = {}   # lives only in THIS process's memory

    def set(self, session_id: str, data: dict) -> None:
        self._sessions[session_id] = data

    def get(self, session_id: str) -> dict | None:
        return self._sessions.get(session_id)


# GOOD: state lives in a backing service (here simulated; in production: Redis)
class ExternalSessionStore:
    def __init__(self, backing_store: dict):
        self._backing_store = backing_store  # stands in for a real Redis connection

    def set(self, session_id: str, data: dict) -> None:
        self._backing_store[session_id] = data

    def get(self, session_id: str) -> dict | None:
        return self._backing_store.get(session_id)

shared_redis_like_store: dict = {}
instance_a = ExternalSessionStore(shared_redis_like_store)
instance_b = ExternalSessionStore(shared_redis_like_store)

instance_a.set("session-1", {"user": "shriman"})
print("read from a different 'instance':", instance_b.get("session-1"))


In [ ]:
 =>  Both 'instances' share the same backing store, so a request landing on instance_b
       still sees session-1 -- this is what actually lets you run more than one instance of
       a service behind a load balancer without users randomly getting logged out.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Audit one of your own services against all 12 factors -- write down which ones it
           genuinely violates today.

 =>  [ ] Take any in-memory cache/session/state in a real project and move it to Redis (or
           at minimum, identify exactly what would break if you ran 2 instances today).


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Storing anything stateful (sessions, uploaded files, rate-limit counters) in process
       memory 'temporarily' -- this quietly becomes permanent and blocks horizontal scaling.

 =>  Treating 12-factor as a checklist to satisfy once, not an ongoing constraint -- a new
       feature can just as easily reintroduce hard-coded config or in-memory state.
